## 9.4 正则化 - 批量归一化Pytorch实现

#### 1. 为什么需要单独学习 BatchNorm 的 PyTorch 实现
在上一小节中，我们已经学习了 Batch Normalization（批量归一化） 的原理：
* 对某一层的输入做标准化
* 再通过 γ 和 β 进行可学习的缩放和平移

它的核心作用是：
* 稳定数据分布
* 加快训练收敛
* 并带来一定正则化效果

但是在真正写代码时，BatchNorm 比 Dropout 更容易让人混淆，主要体现在下面几个问题：
* BatchNorm 在 PyTorch 中到底怎么写
* BatchNorm 应该放在线性层前还是后
* 训练模式和测试模式下 BatchNorm 的行为为什么不同
* BatchNorm1d / 2d / 3d 有什么区别
* running mean 和 running var 是什么

#### 2. BatchNorm 在 PyTorch 中的核心类
PyTorch 为不同类型的数据提供了不同的 BatchNorm 类。

**1️⃣ 常见类**
```
torch.nn.BatchNorm1d
torch.nn.BatchNorm2d
torch.nn.BatchNorm3d
```
它们的区别主要在于：

输入数据的维度不同

**2️⃣ 使用场景**
```
类	常见场景
nn.BatchNorm1d	MLP、全连接层、一维特征
nn.BatchNorm2d	CNN 图像特征图
nn.BatchNorm3d	3D 数据、医学体数据
```

#### 3. BatchNorm1d 的基本写法

##### 3.1 最基本写法
最基础写法
```
import torch.nn as nn
bn = nn.BatchNorm1d(num_features=64)
```
这里的：

`num_features=64`

表示：
* 这一层有 64 个特征维度
* 也就是说，BatchNorm 会对这 64 个特征分别做归一化。

##### 3.2 结合全连接层理解
例如：
```
self.fc1 = nn.Linear(100, 64)
self.bn1 = nn.BatchNorm1d(64)
```
表示：
* fc1 把输入从 100维 变成 64维
* bn1 对这 64维输出 做 BatchNorm

#### 4. BatchNorm 在模型中的标准写法

##### 4.1 推荐顺序
在 MLP / 全连接网络中，最常见、最推荐的顺序是：

`Linear → BatchNorm → Activation`

In [2]:
import torch 
import torch.nn as nn

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(100, 64)
        self.bn = nn.BatchNorm1d(64)
        self.relu = nn.ReLU()
        self.layer_2 = nn.Linear(64, 10)
    
    def forward(self, x):
        x = self.layer_1(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.layer_2(x)
        return x

##### 4.2 为什么通常放在激活函数前
因为 BatchNorm 的主要作用是：
* 对线性层输出做归一化
* 让送入激活函数的数据分布更稳定

所以一般写成：

`Linear → BN → ReLU`

#### 5. 训练模式与测试模式的区别
这是 BatchNorm 在 PyTorch 中最重要的知识点之一。

##### 5.1 训练模式：model.train()
当你写：

`model.train()`

表示模型进入训练模式。

此时 BatchNorm 会：
* 使用当前 batch 的均值和方差

也就是说，每一个 batch 都会单独计算：
* 当前 batch mean
* 当前 batch variance

然后用于标准化。

##### 5.2 测试模式：model.eval()
当你写：

`model.eval()`

表示模型进入评估 / 测试模式。

此时 BatchNorm 不再使用当前 batch 的统计量，而是使用训练过程中累计得到的：
* running mean
* running variance

也就是：
* 全局统计信息的滑动平均

##### 5.3 为什么测试时不能继续用当前 batch 的统计量
因为测试时：
* batch 可能很小
* 甚至只有 1 个样本
* 如果还用当前 batch 的均值和方差，结果会很不稳定

所以测试时更合理的做法是：
* 使用训练阶段累计下来的统计量

#### 6. 什么是 running mean 和 running var
PyTorch 的 BatchNorm 会在训练时维护两个重要变量：
```
running_mean
running_var
```
它们的作用是：
* 记录训练过程中各个 batch 的均值和方差的滑动平均

例如：
* 当前 batch mean = 5
* 下一个 batch mean = 4
* 再下一个 batch mean = 6

PyTorch 会不断更新：

`running_mean`

这样在测试时就可以直接使用。

**查看方式**
```
print(model.bn1.running_mean)
print(model.bn1.running_var)
```

#### 7. BatchNorm 在 PyTorch 中的完整代码示例

##### 7.1 模拟数据
这里模拟一个 10 分类任务：
* 每个样本有 100 个特征
* 标签范围为 0 ~ 9

In [3]:
# 模拟训练数据
X_train = torch.randn(500, 100) # 500个样本，每个样本100维
y_train = torch.randint(0,10, (500,)) # 500个标签，范围在0-9

# 模拟验证数据
X_val = torch.randn(100, 100) # 100个样本，每个样本100维
y_val = torch.randint(0,10, (100,)) # 100个标签，范围在0-9

##### 7.2 构建 DataLoader

In [4]:
from torch.utils.data import TensorDataset, DataLoader
train_ds = TensorDataset(X_train, y_train)
train_dataloader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_ds = TensorDataset(X_val, y_val)
val_dataloader = DataLoader(val_ds, batch_size=32)

##### 7.3 构建模型结构

In [6]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(100, 64) # 全连接层
        self.bn = nn.BatchNorm1d(64) # 批量归一化
        self.relu = nn.ReLU() # 激活函数
        self.layer_2 = nn.Linear(64, 30) # 隐藏层
        self.bn_2 = nn.BatchNorm1d(30) # 批量归一化
        self.relu_2 = nn.ReLU() # 激活函数
        self.layer_3 = nn.Linear(30, 10) # 输出层
    
    def forward(self, x):
        x = self.layer_1(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.layer_2(x)
        x = self.bn_2(x)
        x = self.relu_2(x)
        x = self.layer_3(x)
        return x


##### 7.4 定义损失函数 + 优化器 + scheduler

In [7]:
criterion = nn.CrossEntropyLoss()

model = Model()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer = optimizer,
    mode = 'min',
    factor = 0.1,
    patience = 5,
)

##### 7.5 定义训练方法 + 验证方法

In [8]:
def train(model, train_dataloader, criterion, optimizer):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_dataloader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_dataloader)

def val(model, val_dataloader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_dataloader:
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
    return total_loss / len(val_dataloader)

##### 7.6 训练循环

In [9]:
for epoch in range(100):
    train_loss = train(model, train_dataloader, criterion, optimizer)
    val_loss = val(model, val_dataloader, criterion)
    scheduler.step(val_loss)
    print(f'Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

Epoch 1, Train Loss: 2.3904, Val Loss: 2.3354
Epoch 2, Train Loss: 2.2508, Val Loss: 2.3647
Epoch 3, Train Loss: 2.1482, Val Loss: 2.3796
Epoch 4, Train Loss: 2.0446, Val Loss: 2.3961
Epoch 5, Train Loss: 1.9598, Val Loss: 2.4089
Epoch 6, Train Loss: 1.8774, Val Loss: 2.4231
Epoch 7, Train Loss: 1.8034, Val Loss: 2.4515
Epoch 8, Train Loss: 1.7311, Val Loss: 2.4495
Epoch 9, Train Loss: 1.7198, Val Loss: 2.4579
Epoch 10, Train Loss: 1.6990, Val Loss: 2.4598
Epoch 11, Train Loss: 1.6847, Val Loss: 2.4575
Epoch 12, Train Loss: 1.6768, Val Loss: 2.4603
Epoch 13, Train Loss: 1.6667, Val Loss: 2.4678
Epoch 14, Train Loss: 1.6657, Val Loss: 2.4690
Epoch 15, Train Loss: 1.6586, Val Loss: 2.4682
Epoch 16, Train Loss: 1.6652, Val Loss: 2.4666
Epoch 17, Train Loss: 1.6684, Val Loss: 2.4740
Epoch 18, Train Loss: 1.6586, Val Loss: 2.4736
Epoch 19, Train Loss: 1.6601, Val Loss: 2.4696
Epoch 20, Train Loss: 1.6610, Val Loss: 2.4703
Epoch 21, Train Loss: 1.6701, Val Loss: 2.4734
Epoch 22, Train Loss: 

#### 8. 如何观察 BatchNorm 的效果
可以在训练前后打印：
```
print(model.bn1.running_mean)
print(model.bn1.running_var)
```
这样可以观察：
* 训练过程中 running mean / var 是否在更新

这会帮助你更直观理解：
* BatchNorm 在测试阶段为什么能直接使用全局统计量

#### 9. 使用 BatchNorm 时的常见错误
**1️⃣ 忘记切换 train() / eval()**

这是最常见错误之一。

如果测试时忘了写：

`model.eval()`

那么 BatchNorm 仍然会使用当前 batch 的均值和方差，导致：验证结果不稳定

**2️⃣ BatchNorm1d 的特征维度写错**

例如线性层输出是：

`nn.Linear(100, 64)`

那么 BatchNorm 应该写成：

`nn.BatchNorm1d(64)`

不是 100。

**3️⃣ batch size 太小**

因为 BatchNorm 依赖当前 batch 的均值和方差。

如果 batch size 很小，例如：

`batch_size = 1`

则统计量可能不稳定，效果会变差。